In [11]:
import mlflow
mlflow.set_tracking_uri("file:///home/nico/nico_mle/mlruns")
mlflow.set_experiment("hm-recsys")

/home/nico/nico_mle/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///home/nico/nico_mle/mlruns/369336250363885497', creation_time=1776346502972, experiment_id='369336250363885497', last_update_time=1776346502972, lifecycle_stage='active', name='hm-recsys', tags={}, trace_location=None, workspace='default'>

In [12]:
import scipy.sparse as sp
import numpy as np

user_item    = sp.load_npz("../data/processed/user_item.npz")
customer_idx = np.load("../data/processed/customer_idx.npy", allow_pickle=True)
article_idx  = np.load("../data/processed/article_idx.npy", allow_pickle=True)

In [13]:
print(user_item.shape)                                                                                                                                                                                        
print(customer_idx[:5])                                                                                                                                                                                       
print(article_idx[:5])

(925084, 91875)
['00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657'
 '0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa'
 '000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318'
 '00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a'
 '0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d8cd0c725276a467a2a']
[108775015 108775044 108775051 110065001 110065002]


In [14]:
item_user = user_item.T.tocsr()

In [15]:
print("item_user shape:", item_user.shape)
print("article_idx size:", len(article_idx))
print("user_item shape:", user_item.shape)

item_user shape: (91875, 925084)
article_idx size: 91875
user_item shape: (925084, 91875)


In [16]:
print("item_user shape:", item_user.shape)  # should be (91875, 925084)
print("article_idx size:", len(article_idx))  # should also be 91875

item_user shape: (91875, 925084)
article_idx size: 91875


In [17]:
from implicit.als import AlternatingLeastSquares

with mlflow.start_run(run_name="ALS"):
    model = AlternatingLeastSquares(
        factors=50,
        regularization=0.1,
        iterations=20,
        random_state=42
    )
    
    mlflow.log_params({
        "model": "ALS",
        "factors": 50,
        "regularization": 0.1,
        "iterations": 20
    })
    
    model.fit(user_item)

  0%|          | 0/20 [00:00<?, ?it/s]

In [18]:
from implicit.bpr import BayesianPersonalizedRanking

with mlflow.start_run(run_name="BPR"):
    model_bpr = BayesianPersonalizedRanking(
        factors=50,
        learning_rate=0.01,
        regularization=0.01,
        iterations=100,
        random_state=42
    )
    
    mlflow.log_params({
        "model": "BPR",
        "factors": 50,
        "learning_rate": 0.01,
        "regularization": 0.01,
        "iterations": 100
    })
    
    model_bpr.fit(user_item)

  0%|          | 0/100 [00:00<?, ?it/s]

In [19]:
user_idx = 42

# Who is this user?
print("Customer ID:", customer_idx[user_idx])

# What did they buy?
bought_positions = user_item[user_idx].indices
print("Bought articles:", article_idx[bought_positions].tolist())
print("Purchase counts:", user_item[user_idx].data.tolist())

# What does the model recommend?
ids, scores = model_bpr.recommend(
    user_idx,
    user_item[user_idx],
    N=10,
    filter_already_liked_items=True
)
print("Recommended:", article_idx[ids].tolist())

Customer ID: 0002e6cdaab622b5047407efc0d0bf85e23220e092012095d98119fa9cb3cee7
Bought articles: [189634001, 508691010, 510420008, 552471001, 640021011, 640021019, 647946004, 649046002, 665481013, 673677002, 678625001, 693642001, 735121001, 749615010, 753737015, 783346024, 789303001, 797078014, 812207003, 830953001, 832253001, 835008005, 836699002, 849257001, 849257002, 850622002, 856332001, 857571001, 858883001, 859148001, 862325004, 881562001, 887949002]
Purchase counts: [1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1]
Recommended: [743024003, 766495013, 840947002, 861847004, 766495009, 877462001, 896848001, 831460003, 913720001, 885543004]


In [20]:
ids, scores = model.recommend(
    user_idx,
    user_item[user_idx],
    N=10,
    filter_already_liked_items=True
)
print("ALS Recommended:", article_idx[ids].tolist())

ALS Recommended: [537116001, 673677004, 448509014, 751471001, 179123001, 673677015, 673677010, 768912001, 673677012, 785018003]


In [ ]:
"""
ALS: Tries to give a score to every item and understand how much the customer likes it.
BPR: Tries to rank items according to customers' preferences. User like Item A more than Item B.
Key difference: ALS optimizes prediction accuracy, BPR optimizes ranking order.
For recommendation systems, ranking is what actually matters.
ALS: User will buy 3 of Item A and 1 of Item B
BPR: User prefers Item A over Item B
"""